## 不同阶段训练参数

## 通用参数

In [1]:
import argparse

args = argparse.ArgumentParser()

args.local_rank = 0
args.data_path = ["Dahoas/rm-static"] 
args.data_split = "2,4,4" 
args.data_output_path = '/tmp/data_files'
args.output_dir = "./output_step3_llama"
args.seed = 1234 
args.max_seq_len = 256

## 读取训练数据集

In [2]:
from dschat.utils.data import raw_datasets

dataset_name = "Dahoas/rm-static"

raw_dataset = raw_datasets.DahoasRmstaticDataset(args.output_dir, args.seed, args.local_rank, dataset_name)

train_dataset = raw_dataset.get_train_data()
print(train_dataset)

C:\Users\HP\.conda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Found cached dataset parquet (C:/Users/HP/.cache/huggingface/datasets/Dahoas___parquet/default-b9d2c4937d617106/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)
100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 15.19it/s]

Dataset({
    features: ['prompt', 'response', 'chosen', 'rejected'],
    num_rows: 76256
})


In [3]:
for i, tmp_data in enumerate(train_dataset):
    print(raw_dataset.get_prompt_and_chosen(tmp_data))
    break;



Human: Can you describe the steps to clean fingerprints and smudges from a laptop screen

Assistant: Yes, certainly. To clean your screen, you first need to use a microfiber cloth or soft, damp cloth to gently wipe down the surface of the screen. Next, you’ll want to grab a soft, lint-free, microfiber cleaning cloth and gently rub it back and forth across the screen to remove fingerprints and smudges.

Human: Can I spray isopropyl alcohol onto the cloth and clean it that way?

Assistant: Yes, you can do that to help the cloth pick up even more dirt from the screen. Be sure to always use a clean, soft cloth, not a piece of scratchy, roughened, or textured material, and make sure it’s lint-free.


In [4]:
from transformers import LlamaTokenizer

model_name_or_path = "c:/llama2/llama-2-7b-hf"
tokenizer = LlamaTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

## STEP1 选择数据集（prompt + chosen），合并的训练数据
只显示第一条训练样本

In [5]:
end_of_conversation_token="<|endoftext|>"

prompt_dataset = []
chosen_dataset = []
reject_dataset = []

for i, tmp_data in enumerate(train_dataset):
    # tokenize the text
    chosen_sentence = raw_dataset.get_prompt_and_chosen(tmp_data)  # the accept response
    if chosen_sentence is not None:
        chosen_sentence += end_of_conversation_token
        chosen_token = tokenizer(chosen_sentence,
                                 max_length=args.max_seq_len,
                                 padding="max_length",
                                 truncation=True,
                                 return_tensors="pt")
        chosen_token["input_ids"] = chosen_token["input_ids"].squeeze(0)
        chosen_token["attention_mask"] = chosen_token["attention_mask"].squeeze(0)
        chosen_dataset.append(chosen_token)

    break

print(f'{chosen_dataset[0]=}')

chosen_dataset[0]={'input_ids': tensor([    1, 29871,    13,    13, 29950,  7889, 29901,  1815,   366,  8453,
          278,  6576,   304,  5941, 19917,  2158, 29879,   322,  1560,   566,
         2710,   515,   263, 19022,  4315,    13,    13,  7900, 22137, 29901,
         3869, 29892,  8959, 29889,  1763,  5941,   596,  4315, 29892,   366,
          937,   817,   304,   671,   263,  9200,  7241,   495, 13950,   470,
         4964, 29892,   270,  1160, 13950,   304,   330,  2705,   281, 15705,
         1623,   278,  7101,   310,   278,  4315, 29889,  8084, 29892,   366,
        30010,   645,   864,   304, 17229,   263,  4964, 29892,   301,   524,
        29899,  9021, 29892,  9200,  7241,   495,  5941,   292, 13950,   322,
          330,  2705, 14051,   372,  1250,   322, 11483,  4822,   278,  4315,
          304,  3349, 19917,  2158, 29879,   322,  1560,   566,  2710, 29889,
           13,    13, 29950,  7889, 29901,  1815,   306,   805,   764,   338,
          459,  1336,  2904, 272

## STEP2 选择数据集（prompt + chosen） + 拒绝数据集（prompt + rejected）

In [7]:
end_of_conversation_token="<|endoftext|>"

prompt_dataset = []
chosen_dataset = []
reject_dataset = []

for i, tmp_data in enumerate(train_dataset):
    # tokenize the text
    chosen_sentence = raw_dataset.get_prompt_and_chosen(
        tmp_data)  # the accept response
    reject_sentence = raw_dataset.get_prompt_and_rejected(
        tmp_data)  # the accept response
    if chosen_sentence is not None and reject_sentence is not None:
        chosen_sentence += end_of_conversation_token  # the accept response
        reject_sentence += end_of_conversation_token
        chosen_token = tokenizer(chosen_sentence,
                                 max_length=args.max_seq_len,
                                 padding="max_length",
                                 truncation=True,
                                 return_tensors="pt")
        reject_token = tokenizer(reject_sentence,
                                 max_length=args.max_seq_len,
                                 padding="max_length",
                                 truncation=True,
                                 return_tensors="pt")
        chosen_token["input_ids"] = chosen_token["input_ids"]
        chosen_token["attention_mask"] = chosen_token["attention_mask"]
        chosen_dataset.append(chosen_token)

        reject_token["input_ids"] = reject_token["input_ids"]
        reject_token["attention_mask"] = reject_token["attention_mask"]
        reject_dataset.append(reject_token)
        
    break;

print(f'{chosen_sentence=}')
print(f'{chosen_dataset[0]=}')

print(f'{reject_sentence=}')
print(f'{reject_dataset[0]=}')

chosen_sentence='\n\nHuman: Can you describe the steps to clean fingerprints and smudges from a laptop screen\n\nAssistant: Yes, certainly. To clean your screen, you first need to use a microfiber cloth or soft, damp cloth to gently wipe down the surface of the screen. Next, you’ll want to grab a soft, lint-free, microfiber cleaning cloth and gently rub it back and forth across the screen to remove fingerprints and smudges.\n\nHuman: Can I spray isopropyl alcohol onto the cloth and clean it that way?\n\nAssistant: Yes, you can do that to help the cloth pick up even more dirt from the screen. Be sure to always use a clean, soft cloth, not a piece of scratchy, roughened, or textured material, and make sure it’s lint-free.<|endoftext|>'
chosen_dataset[0]={'input_ids': tensor([[    1, 29871,    13,    13, 29950,  7889, 29901,  1815,   366,  8453,
           278,  6576,   304,  5941, 19917,  2158, 29879,   322,  1560,   566,
          2710,   515,   263, 19022,  4315,    13,    13,  7900, 2

## STEP3 提示数据集（prompt）

In [8]:
filtered = 0
for i, tmp_data in enumerate(train_dataset):
    # tokenize the text
    prompt = raw_dataset.get_prompt(tmp_data)
    if prompt is not None:
        prompt_token = tokenizer(prompt, return_tensors="pt")
        print(f'{prompt_token["input_ids"] =}')
        if prompt_token["input_ids"].size()[-1] <= args.max_seq_len:
            for key_word in ["input_ids", "attention_mask"]:
                #压缩维度[1，139]->[139],再将数据反卷
                prompt_token[key_word] = prompt_token[key_word].squeeze(0).flip(0)
            prompt_dataset.append(prompt_token)
        else:
            filtered += 1

    break;

print(f'{prompt_dataset[0]["input_ids"]=}')

prompt_token["input_ids"] =tensor([[    1, 29871,    13,    13, 29950,  7889, 29901,  1815,   366,  8453,
           278,  6576,   304,  5941, 19917,  2158, 29879,   322,  1560,   566,
          2710,   515,   263, 19022,  4315,    13,    13,  7900, 22137, 29901,
          3869, 29892,  8959, 29889,  1763,  5941,   596,  4315, 29892,   366,
           937,   817,   304,   671,   263,  9200,  7241,   495, 13950,   470,
          4964, 29892,   270,  1160, 13950,   304,   330,  2705,   281, 15705,
          1623,   278,  7101,   310,   278,  4315, 29889,  8084, 29892,   366,
         30010,   645,   864,   304, 17229,   263,  4964, 29892,   301,   524,
         29899,  9021, 29892,  9200,  7241,   495,  5941,   292, 13950,   322,
           330,  2705, 14051,   372,  1250,   322, 11483,  4822,   278,  4315,
           304,  3349, 19917,  2158, 29879,   322,  1560,   566,  2710, 29889,
            13,    13, 29950,  7889, 29901,  1815,   306,   805,   764,   338,
           459,  1336,  2